In [1]:
# =====================================================================
# POKEAPI : get_pokemon_stats(nom_ou_id), avec appel robuste (Q1)
# Lecture seule : aucune écriture sur le disque
# =====================================================================
import time
import requests

BASE = "https://pokeapi.co/api/v2"
CODES_A_REESSAYER = {429, 500, 502, 503, 504}
MESSAGES = {
    400: "Requête invalide (400).",
    401: "Non autorisé (401).",
    403: "Accès interdit (403).",
    404: "Introuvable (404).",
}


def appeler_api(url, params=None, essais=4, pause=1.0):
    """GET robuste (Q1). Renvoie le JSON, None si 404, lève une erreur sinon."""
    for tentative in range(1, essais + 1):
        try:
            rep = requests.get(url, params=params, timeout=10)
        except (requests.ConnectionError, requests.Timeout) as e:
            print(f"  tentative {tentative}/{essais} : {type(e).__name__}")
            attente = pause * 2 ** (tentative - 1)
        else:
            if rep.ok:
                return rep.json()
            if rep.status_code == 404:
                return None
            if rep.status_code not in CODES_A_REESSAYER:
                raise RuntimeError(MESSAGES.get(rep.status_code, f"Erreur HTTP {rep.status_code}"))
            print(f"  tentative {tentative}/{essais} : HTTP {rep.status_code}")
            retry_after = rep.headers.get("Retry-After", "")
            attente = float(retry_after) if retry_after.isdigit() else pause * 2 ** (tentative - 1)
        if tentative < essais:
            time.sleep(attente)
    raise RuntimeError(f"Échec après {essais} tentatives : {url}")


def get_pokemon_stats(nom_ou_id):
    """Renvoie un petit dictionnaire : nom, types et stats. None si le Pokémon n'existe pas."""
    data = appeler_api(f"{BASE}/pokemon/{str(nom_ou_id).strip().lower()}")
    if data is None:
        return None
    # La liste [{"base_stat": 35, "stat": {"name": "hp"}}, ...] devient {"hp": 35, ...}
    stats = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}
    return {
        "id": data["id"],
        "nom": data["name"],
        "types": [t["type"]["name"] for t in data["types"]],
        "pv": stats["hp"],
        "attaque": stats["attack"],
        "defense": stats["defense"],
        "attaque_spe": stats["special-attack"],
        "defense_spe": stats["special-defense"],
        "vitesse": stats["speed"],
    }


print(get_pokemon_stats("Pikachu"))     # par nom (la casse n'a pas d'importance)
print(get_pokemon_stats(6))             # par id
print(get_pokemon_stats("inconnumon"))  # attendu : None

{'id': 25, 'nom': 'pikachu', 'types': ['electric'], 'pv': 35, 'attaque': 55, 'defense': 40, 'attaque_spe': 50, 'defense_spe': 50, 'vitesse': 90}
{'id': 6, 'nom': 'charizard', 'types': ['fire', 'flying'], 'pv': 78, 'attaque': 84, 'defense': 78, 'attaque_spe': 109, 'defense_spe': 85, 'vitesse': 100}
None


In [2]:
# =====================================================================
# POKEAPI : comparer deux Pokémon (PV et attaque)
# Prérequis : cellule précédente exécutée (get_pokemon_stats)
# =====================================================================
import pandas as pd


def comparer_pokemon(nom_a, nom_b):
    """Compare deux Pokémon sur les PV et l'attaque. Renvoie le nom du plus fort, 'égalité' ou None."""
    a, b = get_pokemon_stats(nom_a), get_pokemon_stats(nom_b)
    if a is None or b is None:
        inconnus = [n for n, p in ((nom_a, a), (nom_b, b)) if p is None]
        print("Pokémon introuvable :", ", ".join(inconnus))
        return None

    # Tableau de comparaison
    tableau = pd.DataFrame({a["nom"]: a, b["nom"]: b}).loc[["pv", "attaque", "defense", "vitesse"]]
    print(tableau.to_string(), "\n")

    # Points : 1 point par critère gagné (PV et attaque seulement, comme l'énoncé)
    points = {a["nom"]: 0, b["nom"]: 0}
    for critere in ("pv", "attaque"):
        if a[critere] > b[critere]:
            points[a["nom"]] += 1
            print(f"{critere:8s}: {a['nom']} gagne ({a[critere]} contre {b[critere]})")
        elif b[critere] > a[critere]:
            points[b["nom"]] += 1
            print(f"{critere:8s}: {b['nom']} gagne ({b[critere]} contre {a[critere]})")
        else:
            print(f"{critere:8s}: égalité ({a[critere]})")

    if points[a["nom"]] > points[b["nom"]]:
        verdict = a["nom"]
    elif points[b["nom"]] > points[a["nom"]]:
        verdict = b["nom"]
    else:
        verdict = "égalité"
    print(f"\nVerdict : {verdict}")
    return verdict


comparer_pokemon("pikachu", "charizard")     # charizard devrait l'emporter sur les deux critères
print("-" * 40)
comparer_pokemon("pikachu", "inconnumon")    # attendu : message d'erreur, pas de plantage

        pikachu charizard
pv           35        78
attaque      55        84
defense      40        78
vitesse      90       100 

pv      : charizard gagne (78 contre 35)
attaque : charizard gagne (84 contre 55)

Verdict : charizard
----------------------------------------
Pokémon introuvable : inconnumon


In [3]:
# =====================================================================
# POKEAPI : stats par type (nombre + moyenne des PV) avec asyncio (Q2) et cache 1 h (Q3)
# Lecture seule : aucune écriture sur le disque
# =====================================================================
import asyncio
import time
import aiohttp

cache_pokemon = {}      # url -> (heure, pv) : cache d'1 heure
TTL = 3600


async def pv_pokemon(session, sem, url):
    """Renvoie les PV d'un Pokémon (cache d'1 h, 20 appels simultanés au maximum)."""
    if url in cache_pokemon and time.time() - cache_pokemon[url][0] < TTL:
        return cache_pokemon[url][1]
    async with sem:
        for tentative in range(1, 5):
            try:
                async with session.get(url, timeout=aiohttp.ClientTimeout(total=15)) as rep:
                    if rep.status == 200:
                        data = await rep.json()
                        pv = next(s["base_stat"] for s in data["stats"] if s["stat"]["name"] == "hp")
                        cache_pokemon[url] = (time.time(), pv)
                        return pv
                    if rep.status == 404:
                        return None
            except (aiohttp.ClientError, asyncio.TimeoutError):
                pass
            await asyncio.sleep(2 ** (tentative - 1))    # attente croissante, puis nouvelle tentative
    return None


async def stats_par_type(type_pokemon):
    """Nombre de Pokémon du type et moyenne de leurs PV."""
    data = appeler_api(f"{BASE}/type/{type_pokemon.strip().lower()}")
    if data is None:
        print("Type introuvable :", type_pokemon)
        return None
    urls = [p["pokemon"]["url"] for p in data["pokemon"]]
    sem = asyncio.Semaphore(20)
    async with aiohttp.ClientSession() as session:
        pvs = await asyncio.gather(*(pv_pokemon(session, sem, u) for u in urls))
    valides = [pv for pv in pvs if pv is not None]
    return {"type": type_pokemon, "nombre": len(urls), "pv_moyen": round(sum(valides) / len(valides), 1),
            "echecs": len(urls) - len(valides)}


for t in ("dragon", "fire"):
    debut = time.time()
    print(await stats_par_type(t), f"| {time.time() - debut:.1f} s")

debut = time.time()
print(await stats_par_type("dragon"), f"| {time.time() - debut:.1f} s (cache)")

{'type': 'dragon', 'nombre': 117, 'pv_moyen': 89.7, 'echecs': 0} | 1.1 s
{'type': 'fire', 'nombre': 109, 'pv_moyen': 72.5, 'echecs': 0} | 0.9 s
{'type': 'dragon', 'nombre': 117, 'pv_moyen': 89.7, 'echecs': 0} | 0.3 s (cache)


In [4]:
# =====================================================================
# POKEAPI : combat sur 5 tours (dégâts totaux)
# Prérequis : get_pokemon_stats exécutée
# =====================================================================
def degats(attaquant, defenseur):
    """Dégâts d'une attaque : attaque - défense/2, minimum 1."""
    return max(1, attaquant["attaque"] - defenseur["defense"] / 2)


def combat(nom_a, nom_b, tours=5):
    a, b = get_pokemon_stats(nom_a), get_pokemon_stats(nom_b)
    if a is None or b is None:
        print("Pokémon introuvable")
        return None
    total = {a["nom"]: 0.0, b["nom"]: 0.0}
    for t in range(1, tours + 1):
        da, db = degats(a, b), degats(b, a)      # stats de base : dégâts identiques à chaque tour
        total[a["nom"]] += da
        total[b["nom"]] += db
        print(f"Tour {t} : {a['nom']} inflige {da:.1f} | {b['nom']} inflige {db:.1f}")
    print("Totaux :", {k: round(v, 1) for k, v in total.items()})
    if total[a["nom"]] == total[b["nom"]]:
        print("Résultat : match nul")
        return "nul"
    gagnant = max(total, key=total.get)
    print("Vainqueur :", gagnant)
    return gagnant


combat("pikachu", "charizard")

Tour 1 : pikachu inflige 16.0 | charizard inflige 64.0
Tour 2 : pikachu inflige 16.0 | charizard inflige 64.0
Tour 3 : pikachu inflige 16.0 | charizard inflige 64.0
Tour 4 : pikachu inflige 16.0 | charizard inflige 64.0
Tour 5 : pikachu inflige 16.0 | charizard inflige 64.0
Totaux : {'pikachu': 80.0, 'charizard': 320.0}
Vainqueur : charizard


'charizard'